# Step 4 — Cortex AI Enrichment

We use three built-in Cortex AI functions to add intelligence to our payments data — no model training, no infrastructure, just SQL.

| Function | Purpose |
|---|---|
| `SENTIMENT()` | Score feedback text –1 to +1 |
| `CLASSIFY_TEXT()` | Assign a topic label from a list |
| `COMPLETE()` | Prompt an LLM for free-text output |

In [ ]:
%%sql -r ctx
USE WAREHOUSE BARCLAYS_WH;
USE SCHEMA BARCLAYS_HOL.ANALYTICS;

---
## Enrich customer feedback with sentiment and topic

In [ ]:
%%sql -r enriched_result
CREATE OR REPLACE TABLE FEEDBACK_ENRICHED AS
WITH ai_scored AS (
    SELECT
        f.*,
        c.CUSTOMER_NAME,
        c.SECTOR            AS CUSTOMER_SECTOR,
        c.RM_NAME,
        ROUND(SNOWFLAKE.CORTEX.SENTIMENT(f.FEEDBACK_TEXT), 3)
            AS SENTIMENT_SCORE,
        SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
            f.FEEDBACK_TEXT,
            ['Performance Issue','Service Quality','Technical Problem',
             'Positive Experience','Feature Request']
        ) AS CLASSIFICATION_RESULT
    FROM STG_FEEDBACK f
    LEFT JOIN STG_CUSTOMERS c ON f.CUSTOMER_ID = c.CUSTOMER_ID
)
SELECT
    FEEDBACK_ID, FEEDBACK_DATE, CUSTOMER_ID, CUSTOMER_NAME,
    CUSTOMER_SECTOR, RM_NAME, PAYMENT_ID, FEEDBACK_CHANNEL,
    FEEDBACK_TEXT, PAYMENT_TYPE, RATING, SENTIMENT_SCORE,
    CASE
        WHEN SENTIMENT_SCORE >  0.3 THEN 'POSITIVE'
        WHEN SENTIMENT_SCORE < -0.3 THEN 'NEGATIVE'
        ELSE 'NEUTRAL'
    END AS SENTIMENT_LABEL,
    CLASSIFICATION_RESULT:label::STRING  AS TOPIC_CATEGORY,
    CLASSIFICATION_RESULT:score::FLOAT   AS CLASSIFICATION_CONFIDENCE
FROM ai_scored;

In [ ]:
%%sql -r feedback_sample
SELECT CUSTOMER_NAME, CUSTOMER_SECTOR, LEFT(FEEDBACK_TEXT, 80) AS FEEDBACK_PREVIEW,
       SENTIMENT_SCORE, SENTIMENT_LABEL, TOPIC_CATEGORY
FROM FEEDBACK_ENRICHED LIMIT 10;

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = feedback_sample.to_pandas()

# Sentiment score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: sentiment label counts
label_counts = df.groupby('SENTIMENT_LABEL').size().reindex(['NEGATIVE','NEUTRAL','POSITIVE'])
colours = ['#e74c3c', '#95a5a6', '#2ecc71']
label_counts.plot.bar(ax=axes[0], color=colours, edgecolor='white')
axes[0].set_title('Sentiment Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Right: topic category counts
df['TOPIC_CATEGORY'].value_counts().plot.barh(ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Topic Classification')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

---
## Analyse sentiment across the full dataset

In [ ]:
%%sql -r sentiment_by_type
SELECT
    PAYMENT_TYPE,
    COUNT(*) AS TOTAL,
    ROUND(AVG(SENTIMENT_SCORE), 3) AS AVG_SENTIMENT,
    COUNT(CASE WHEN SENTIMENT_LABEL = 'NEGATIVE' THEN 1 END) AS NEGATIVES,
    ROUND(COUNT(CASE WHEN SENTIMENT_LABEL = 'NEGATIVE' THEN 1 END)::FLOAT
        / COUNT(*) * 100, 1) AS NEGATIVE_PCT
FROM FEEDBACK_ENRICHED
GROUP BY PAYMENT_TYPE
ORDER BY AVG_SENTIMENT ASC;

In [ ]:
df = sentiment_by_type.to_pandas()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(df['PAYMENT_TYPE'], df['AVG_SENTIMENT'],
               color=['#e74c3c' if s < 0 else '#2ecc71' for s in df['AVG_SENTIMENT']],
               edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Average Sentiment Score')
ax.set_title('Average Sentiment by Payment Scheme')
for bar, pct in zip(bars, df['NEGATIVE_PCT']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{pct}% neg', va='center', fontsize=9, color='#666')
plt.tight_layout()
plt.show()

---
## Generate recommended actions for critical alerts

In [ ]:
%%sql -r alert_recs
-- Pick alerts that have error context first, then fill remaining slots without
WITH ranked_alerts AS (
    SELECT
        a.ALERT_ID, a.SEVERITY, a.ALERT_DESCRIPTION, a.PAYMENT_TYPE,
        e.ERROR_DESCRIPTION, e.ERROR_CATEGORY, e.IS_RETRYABLE,
        ROW_NUMBER() OVER (
            ORDER BY IFF(e.ERROR_DESCRIPTION IS NOT NULL, 0, 1), a.ALERT_TIME DESC
        ) AS RN
    FROM BARCLAYS_HOL.RAW_DATA.OPERATIONAL_ALERTS a
    LEFT JOIN BARCLAYS_HOL.RAW_DATA.PAYMENTS p ON a.PAYMENT_ID = p.PAYMENT_ID
    LEFT JOIN BARCLAYS_HOL.RAW_DATA.ERROR_CODES e ON p.ERROR_CODE = e.ERROR_CODE
    WHERE a.STATUS = 'OPEN' AND a.SEVERITY IN ('HIGH','CRITICAL')
)
SELECT
    ALERT_ID, SEVERITY, ALERT_DESCRIPTION, PAYMENT_TYPE, ERROR_DESCRIPTION,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-8b',
        'You are a payments operations analyst at a UK bank. '
        || 'Alert: "' || ALERT_DESCRIPTION || '" '
        || 'Payment type: ' || PAYMENT_TYPE || '. '
        || 'Severity: ' || SEVERITY || '. '
        || IFF(ERROR_DESCRIPTION IS NOT NULL,
               'Root cause: ' || ERROR_DESCRIPTION
               || ' (' || ERROR_CATEGORY || ', retryable: ' || IS_RETRYABLE::VARCHAR || '). ', '')
        || 'Respond with ONE sentence: the recommended action.'
    ) AS AI_RECOMMENDED_ACTION
FROM ranked_alerts
WHERE RN <= 5;

---
## Dynamic Tables — real-time operational layer

Dynamic Tables declare a query and a **target lag** — Snowflake keeps the results fresh automatically. No tasks, no scheduling, no orchestration code.

- `TARGET_LAG = DOWNSTREAM` — refreshes only when a downstream consumer needs it
- `TARGET_LAG = '1 minute'` — Snowflake guarantees results are never more than 1 minute stale

### DT 1: Negative Sentiment Feed

**Who uses it:** Customer Experience ops team, escalation dashboards

**Purpose:** Surfaces all negative-sentiment feedback with the customer name, sector, and assigned RM — so the ops team can triage complaints and route them to the right relationship manager without running ad-hoc queries.

**Refresh:** `DOWNSTREAM` — only materialises when queried, keeping costs low.

In [ ]:
%%sql -r dt1
CREATE OR REPLACE DYNAMIC TABLE DT_NEGATIVE_SENTIMENT_FEED
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = BARCLAYS_WH
AS
SELECT FEEDBACK_ID, FEEDBACK_DATE, CUSTOMER_NAME, CUSTOMER_SECTOR,
       RM_NAME, PAYMENT_TYPE, FEEDBACK_TEXT, SENTIMENT_SCORE, TOPIC_CATEGORY
FROM FEEDBACK_ENRICHED
WHERE SENTIMENT_LABEL = 'NEGATIVE'
ORDER BY FEEDBACK_DATE DESC;

### DT 2: Sentiment Summary by Payment Type

**Who uses it:** Payments product managers, weekly review decks

**Purpose:** Aggregates feedback counts and average sentiment per payment scheme. Lets product owners spot which schemes are generating the most complaints — e.g. if BACS sentiment drops, it may indicate a processing delay affecting that rail.

**Refresh:** `DOWNSTREAM` — cascades from `FEEDBACK_ENRICHED`, refreshes on demand.

In [ ]:
%%sql -r dt2
CREATE OR REPLACE DYNAMIC TABLE DT_SENTIMENT_OPS_SUMMARY
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = BARCLAYS_WH
AS
SELECT PAYMENT_TYPE, COUNT(*) AS TOTAL_FEEDBACK,
       COUNT(CASE WHEN SENTIMENT_LABEL = 'NEGATIVE' THEN 1 END) AS NEGATIVE_COUNT,
       COUNT(CASE WHEN SENTIMENT_LABEL = 'POSITIVE' THEN 1 END) AS POSITIVE_COUNT,
       ROUND(AVG(SENTIMENT_SCORE), 3) AS AVG_SENTIMENT
FROM FEEDBACK_ENRICHED
GROUP BY PAYMENT_TYPE;

### DT 3: Live SLA Breach Monitor

**Who uses it:** Payments operations, real-time monitoring screens

**Purpose:** Shows the volume of payments that breached SLA per scheme in the last 7 days — total payments, breached count, and breach rate. This is the view on the ops floor dashboard — if CHAPS shows 500 breached payments at 12%, someone investigates immediately.

**Refresh:** `1 minute` — the only DT with an active refresh cadence because SLA breaches are time-critical.

In [ ]:
%%sql -r dt3
CREATE OR REPLACE DYNAMIC TABLE DT_LIVE_SLA_BREACHES
    TARGET_LAG = '1 minute'
    WAREHOUSE = BARCLAYS_WH
AS
SELECT
    PAYMENT_TYPE,
    SUM(TOTAL_PAYMENTS) AS TOTAL_PAYMENTS,
    SUM(FAILED_PAYMENTS) AS BREACHED_PAYMENTS,
    ROUND(SUM(FAILED_PAYMENTS)::FLOAT / NULLIF(SUM(TOTAL_PAYMENTS), 0) * 100, 2) AS BREACH_RATE_PCT,
    MIN(PAYMENT_DAY) AS EARLIEST_BREACH,
    MAX(PAYMENT_DAY) AS LATEST_BREACH
FROM MART_SLA_COMPLIANCE
WHERE SLA_STATUS = 'BREACH'
  AND PAYMENT_DAY >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY PAYMENT_TYPE;

In [ ]:
%%sql -r dt_status
SHOW DYNAMIC TABLES IN SCHEMA ANALYTICS;